In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

from load import load_data, tensorize_data, clean_data
from models import NaiveRNN, NaiveLSTM
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
def prepareDataLoader(dataset, labels, batchsize=32):
    train, test, train_labels, test_labels = train_test_split(dataset, labels, test_size=0.2, random_state=42)


    train_dataset = TensorDataset(train, train_labels)
    test_dataset = TensorDataset(test, test_labels)

    train_dataloader = DataLoader(train_dataset, batch_size=batchsize, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batchsize, shuffle=False)

    return (train_dataloader, test_dataloader)

In [3]:
def training(epochs, model, dataloader, criterion, optimizer):
    for epoch in range(epochs):
        model.train()
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

def evaluate(model, dataloader):
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f'Accuracy: {100 * correct / total}%')


In [4]:
dfs = load_data()

# The load_data function will return dfs in a different order. Parse through
for df in dfs:
    if df[1] == "PFC_con_4.csv":
        con4_df = df[0]
    if df[1] == "PFC_con_5.csv":
        con5_df = df[0]

In [5]:
# Use DS+/DS- as labels (0)
tensors_4 = tensorize_data(con4_df, 0)
labels = tensors_4[1]
dataset = tensors_4[0]

# Unsqueeze my tensor bc it think I'm using a forward hook or something???
unsqueezed_tensor = dataset.unsqueeze(-1)

dataloaders = prepareDataLoader(unsqueezed_tensor, labels, 32)
train_dataloader = dataloaders[0]
test_dataloader = dataloaders[1]

In [6]:
rnn_model = NaiveRNN(1, 32, 2).to(device)
lstm_model = NaiveLSTM(1, 32, 2).to(device)

criterion = nn.CrossEntropyLoss()
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.001)
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)


epochs = 50
training(epochs, rnn_model, train_dataloader, criterion, rnn_optimizer)
evaluate(rnn_model, test_dataloader)

Epoch [1/50], Loss: 0.7142
Epoch [2/50], Loss: 0.6969
Epoch [3/50], Loss: 0.7086
Epoch [4/50], Loss: 0.6826
Epoch [5/50], Loss: 0.6896
Epoch [6/50], Loss: 0.6757
Epoch [7/50], Loss: 0.6910
Epoch [8/50], Loss: 0.7012
Epoch [9/50], Loss: 0.6977
Epoch [10/50], Loss: 0.7052
Epoch [11/50], Loss: 0.6826
Epoch [12/50], Loss: 0.6808
Epoch [13/50], Loss: 0.6505
Epoch [14/50], Loss: 0.6866
Epoch [15/50], Loss: 0.7007
Epoch [16/50], Loss: 0.6890
Epoch [17/50], Loss: 0.6779
Epoch [18/50], Loss: 0.6856
Epoch [19/50], Loss: 0.6816
Epoch [20/50], Loss: 0.6909
Epoch [21/50], Loss: 0.6864
Epoch [22/50], Loss: 0.6669
Epoch [23/50], Loss: 0.6806
Epoch [24/50], Loss: 0.7090
Epoch [25/50], Loss: 0.6657
Epoch [26/50], Loss: 0.6923
Epoch [27/50], Loss: 0.7207
Epoch [28/50], Loss: 0.6908
Epoch [29/50], Loss: 0.6612
Epoch [30/50], Loss: 0.6828
Epoch [31/50], Loss: 0.6936
Epoch [32/50], Loss: 0.6967
Epoch [33/50], Loss: 0.6934
Epoch [34/50], Loss: 0.6784
Epoch [35/50], Loss: 0.6874
Epoch [36/50], Loss: 0.6720
E

In [7]:
con4_df_clean = clean_data(con4_df)
# Use DS+/DS- as labels (0)
dataset, labels = tensorize_data(con4_df_clean, 0)

# Unsqueeze my tensor bc it think I'm using a forward hook or something???
unsqueezed_tensor = dataset.unsqueeze(-1)

dataloaders = prepareDataLoader(unsqueezed_tensor, labels, 32)
train_dataloader = dataloaders[0]
test_dataloader = dataloaders[1]

epochs = 20
rnn_model = NaiveRNN(1, 32, 2).to(device)
training(epochs, rnn_model, train_dataloader, criterion, rnn_optimizer)
evaluate(rnn_model, test_dataloader)

Epoch [1/20], Loss: 0.7088
Epoch [2/20], Loss: 0.6627
Epoch [3/20], Loss: 0.7095
Epoch [4/20], Loss: 0.6855
Epoch [5/20], Loss: 0.7201
Epoch [6/20], Loss: 0.7410
Epoch [7/20], Loss: 0.6973
Epoch [8/20], Loss: 0.7162
Epoch [9/20], Loss: 0.6994
Epoch [10/20], Loss: 0.7181
Epoch [11/20], Loss: 0.6984
Epoch [12/20], Loss: 0.6830
Epoch [13/20], Loss: 0.6932
Epoch [14/20], Loss: 0.7041
Epoch [15/20], Loss: 0.7174
Epoch [16/20], Loss: 0.6769
Epoch [17/20], Loss: 0.6964
Epoch [18/20], Loss: 0.6830
Epoch [19/20], Loss: 0.7144
Epoch [20/20], Loss: 0.7202
Accuracy: 49.357226334242306%


In [8]:
ds_minus = con4_df_clean[con4_df_clean.iloc[:, 3] == 0]
dataset, labels = tensorize_data(ds_minus, 0)

unsqueezed_tensor = dataset.unsqueeze(-1)

train, test = prepareDataLoader(unsqueezed_tensor, labels, 32)

criterion = nn.CrossEntropyLoss()
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.01)
epochs = 20
rnn_model = NaiveRNN(1, 32, 2).to(device)


training(epochs, rnn_model, train_dataloader, criterion, rnn_optimizer)
evaluate(rnn_model, test_dataloader)

Epoch [1/20], Loss: 0.7280
Epoch [2/20], Loss: 0.7049
Epoch [3/20], Loss: 0.6943
Epoch [4/20], Loss: 0.6816
Epoch [5/20], Loss: 0.7040
Epoch [6/20], Loss: 0.7194
Epoch [7/20], Loss: 0.6689
Epoch [8/20], Loss: 0.6771
Epoch [9/20], Loss: 0.6795
Epoch [10/20], Loss: 0.7052
Epoch [11/20], Loss: 0.6874
Epoch [12/20], Loss: 0.7220
Epoch [13/20], Loss: 0.6727
Epoch [14/20], Loss: 0.7525
Epoch [15/20], Loss: 0.7043
Epoch [16/20], Loss: 0.6910
Epoch [17/20], Loss: 0.7125
Epoch [18/20], Loss: 0.7148
Epoch [19/20], Loss: 0.7134
Epoch [20/20], Loss: 0.7115
Accuracy: 49.3961823139852%


In [9]:
ds_minus = con4_df_clean[con4_df_clean.iloc[:, 3] == 0]
dataset, labels = tensorize_data(ds_minus, 0)


In [10]:
# Investigate why this is only achieving ~50% accuracy but my test is achieving like 77%

In [11]:
import os
con4_df = pd.read_csv(os.path.join("./Neuron Data", "PFC_con_4.csv"))
con4_df = con4_df.apply(pd.to_numeric, errors='coerce')

con4_df = clean_data(con4_df)

# Drop rat number, cell number, trial number
con4_df = con4_df.drop(columns=con4_df.columns[:3])
con4_df.columns = range(len(con4_df.columns))

# Split by trial type
con4_minus = con4_df[con4_df.iloc[:, 0] == 0]
con4_minus = con4_minus.drop(columns=con4_minus.columns[0])

# Extract labels
con4_minus_labels = con4_minus.iloc[:, 0].tolist()
con4_minus_labels = torch.tensor(con4_minus_labels, dtype=torch.long)

con4_minus = con4_minus.drop(columns=con4_minus.columns[0])

# Convert to tensors
con4_minus_tensor = torch.tensor(con4_minus.to_numpy(), dtype=torch.float32)


In [12]:
from sklearn.model_selection import train_test_split

# Trying to unsqueeze the tensor
con4_minus_tensor = torch.tensor(con4_minus.to_numpy(), dtype=torch.float32)
con4_minus_tensor = con4_minus_tensor.unsqueeze(-1)

train, test = prepareDataLoader(con4_minus_tensor, con4_minus_labels, 32)

criterion = nn.CrossEntropyLoss()
rnn_model = NaiveRNN(1, 32, 2).to(device)
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.001)
epochs = 20

training(epochs, rnn_model, train, criterion, rnn_optimizer)
evaluate(rnn_model, test)

Epoch [1/20], Loss: 0.2375
Epoch [2/20], Loss: 0.3816
Epoch [3/20], Loss: 0.2377
Epoch [4/20], Loss: 0.3936
Epoch [5/20], Loss: 0.3779
Epoch [6/20], Loss: 0.2292
Epoch [7/20], Loss: 0.3650
Epoch [8/20], Loss: 0.4909
Epoch [9/20], Loss: 0.2276
Epoch [10/20], Loss: 0.6336
Epoch [11/20], Loss: 0.4808
Epoch [12/20], Loss: 0.3637
Epoch [13/20], Loss: 0.7773
Epoch [14/20], Loss: 0.4886
Epoch [15/20], Loss: 0.5087
Epoch [16/20], Loss: 0.8267
Epoch [17/20], Loss: 0.5371
Epoch [18/20], Loss: 0.5901
Epoch [19/20], Loss: 0.7469
Epoch [20/20], Loss: 0.4715
Accuracy: 77.9423226812159%


In [13]:
# Repeat just using base functionality
con4_df = pd.read_csv(os.path.join("./Neuron Data", "PFC_con_4.csv"))
con4_df = clean_data(con4_df)

con4_tensor, con4_labels = tensorize_data(con4_df, 1)

con4_tensor_unsqueeze = con4_tensor.unsqueeze(-1)

train, test = prepareDataLoader(con4_tensor_unsqueeze, con4_labels, 32)

criterion = nn.CrossEntropyLoss()
rnn_model = NaiveRNN(1, 32, 2).to(device)
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.001)
epochs = 20

training(epochs, rnn_model, train, criterion, rnn_optimizer)
evaluate(rnn_model, test)

Epoch [1/20], Loss: 0.6998
Epoch [2/20], Loss: 0.6567
Epoch [3/20], Loss: 0.7024
Epoch [4/20], Loss: 0.6538
Epoch [5/20], Loss: 0.6644
Epoch [6/20], Loss: 0.6692
Epoch [7/20], Loss: 0.6599
Epoch [8/20], Loss: 0.6475
Epoch [9/20], Loss: 0.6697
Epoch [10/20], Loss: 0.6446
Epoch [11/20], Loss: 0.7123
Epoch [12/20], Loss: 0.6525
Epoch [13/20], Loss: 0.6619
Epoch [14/20], Loss: 0.7278
Epoch [15/20], Loss: 0.6660
Epoch [16/20], Loss: 0.7206
Epoch [17/20], Loss: 0.6866
Epoch [18/20], Loss: 0.6861
Epoch [19/20], Loss: 0.7026
Epoch [20/20], Loss: 0.6853
Accuracy: 61.35566809505259%


In [14]:
lstm_model = NaiveLSTM(1, 32, 2).to(device)
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)
epochs = 20

training(epochs, lstm_model, train, criterion, lstm_optimizer)
evaluate(lstm_model, test)

Epoch [1/20], Loss: 0.7208
Epoch [2/20], Loss: 0.6728
Epoch [3/20], Loss: 0.6918
Epoch [4/20], Loss: 0.6798
Epoch [5/20], Loss: 0.6545
Epoch [6/20], Loss: 0.6807
Epoch [7/20], Loss: 0.6688
Epoch [8/20], Loss: 0.6060
Epoch [9/20], Loss: 0.6777
Epoch [10/20], Loss: 0.6343
Epoch [11/20], Loss: 0.6321
Epoch [12/20], Loss: 0.6703
Epoch [13/20], Loss: 0.6559
Epoch [14/20], Loss: 0.6487
Epoch [15/20], Loss: 0.6455
Epoch [16/20], Loss: 0.6376
Epoch [17/20], Loss: 0.6617
Epoch [18/20], Loss: 0.6690
Epoch [19/20], Loss: 0.6403
Epoch [20/20], Loss: 0.6495
Accuracy: 61.23880015582392%
